## 7. Class Imbalance Handling--------------------------------------------------

# COST-SENSITIVE

In [ ]:
# Cost-Sensitive Learning (penalizes misclassification of minority classes during training)
# class_weight='balanced' automatically computes weights inversely proportional to class frequency

# Cost-Sensitive w Decision Tree
cs_dt_pipe = SkPipeline([
    ('preprocess', preprocess),
    ('clf', DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE))
])
cs_dt_grid = GridSearchCV(
    cs_dt_pipe,
    param_grid={
        'clf__max_depth': [10, 20, 30, None],
        'clf__min_samples_split': [2, 5, 10],
        'clf__criterion': ['gini', 'entropy']
    },
    scoring=SCORING, cv=cv, n_jobs=-1, verbose=1
)
cs_dt_grid.fit(X_train, y_train)
cs_dt_best = cs_dt_grid.best_estimator_
cs_dt_metrics = evaluate('Cost-Sensitive DT \u2014 VALIDATION', cs_dt_best, X_val, y_val)
results_table.append({'Model': 'Cost-Sensitive Decision Tree', **cs_dt_metrics})

Fitting 5 folds for each of 24 candidates, totalling 120 fits

=== Cost-Sensitive DT — VALIDATION ===
              precision    recall  f1-score   support

           1      0.935     0.930     0.932     42368
           2      0.940     0.945     0.943     56660
           3      0.917     0.914     0.915      7151
           4      0.846     0.851     0.848       549
           5      0.831     0.807     0.819      1899
           6      0.856     0.854     0.855      3474
           7      0.946     0.943     0.945      4102

    accuracy                          0.932    116203
   macro avg      0.896     0.892     0.894    116203
weighted avg      0.932     0.932     0.932    116203

Accuracy         : 0.9322
Balanced Accuracy: 0.8919
F1 Macro         : 0.8938

Per-Class Recall:
  Class 1: 0.9301
  Class 2: 0.9451
  Class 3: 0.9139
  Class 4: 0.8506
  Class 5: 0.8067
  Class 6: 0.8538
  Class 7: 0.9434

Confusion Matrix:
[[39407  2710     7     0    39    13   192]
 [ 2492 53548 

In [ ]:
# Cost-Sensitive w Random Forest
cs_rf_pipe = SkPipeline([
    ('preprocess', preprocess),
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))
])
cs_rf_search = RandomizedSearchCV(
    cs_rf_pipe,
    param_distributions={
        'clf__n_estimators': [100, 200, 300],
        'clf__max_depth': [10, 20, None],
        'clf__min_samples_split': [2, 5],
        'clf__max_features': ['sqrt', 'log2']
    },
    n_iter=20, scoring=SCORING, cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=1
)
cs_rf_search.fit(X_train, y_train)
cs_rf_best = cs_rf_search.best_estimator_
cs_rf_metrics = evaluate('Cost-Sensitive RF \u2014 VALIDATION', cs_rf_best, X_val, y_val)
results_table.append({'Model': 'Cost-Sensitive Random Forest', **cs_rf_metrics})

Fitting 5 folds for each of 20 candidates, totalling 100 fits

=== Cost-Sensitive RF — VALIDATION ===
              precision    recall  f1-score   support

           1      0.959     0.933     0.946     42368
           2      0.945     0.965     0.955     56660
           3      0.929     0.955     0.942      7151
           4      0.870     0.902     0.886       549
           5      0.894     0.799     0.844      1899
           6      0.892     0.901     0.897      3474
           7      0.967     0.953     0.960      4102

    accuracy                          0.947    116203
   macro avg      0.922     0.915     0.918    116203
weighted avg      0.947     0.947     0.947    116203

Accuracy         : 0.9471
Balanced Accuracy: 0.9153
F1 Macro         : 0.9183

Per-Class Recall:
  Class 1: 0.9325
  Class 2: 0.9647
  Class 3: 0.9553
  Class 4: 0.9016
  Class 5: 0.7994
  Class 6: 0.9007
  Class 7: 0.9532

Confusion Matrix:
[[39510  2706     3     0    26    14   109]
 [ 1477 54658 

In [ ]:
# Cost-Sensitive w XGBoost
# XGBoost does not have a native class_weight parameter
# Instead, I compute sample weights manually and pass them during fitting
w_train = compute_sample_weight(class_weight='balanced', y=y_train_adj)
cs_xgb_pipe = SkPipeline([
    ('prep', preprocess),
    ('clf', XGBClassifier(
        objective='multi:softprob', num_class=num_classes,
        eval_metric='mlogloss', tree_method='hist',
        random_state=RANDOM_STATE, n_jobs=-1
    ))
])
cs_xgb_search = RandomizedSearchCV(
    cs_xgb_pipe,
    param_distributions={
        'clf__n_estimators': [100, 200, 300],
        'clf__max_depth': [4, 6, 8],
        'clf__learning_rate': [0.05, 0.1, 0.2]
    },
    n_iter=20, scoring=SCORING, cv=cv,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=1
)
try:
    cs_xgb_search.fit(X_train, y_train_adj, clf__sample_weight=w_train)
except TypeError:
    # Fallback if fit without weights then refit best estimator with weights
    cs_xgb_search.fit(X_train, y_train_adj)
    cs_xgb_search.best_estimator_.fit(X_train, y_train_adj, clf__sample_weight=w_train)
cs_xgb_best = cs_xgb_search.best_estimator_
cs_xgb_metrics = evaluate('Cost-Sensitive XGB \u2014 VALIDATION', cs_xgb_best, X_val, y_val_adj, offset=1)
results_table.append({'Model': 'Cost-Sensitive XGBoost', **cs_xgb_metrics})

Fitting 5 folds for each of 20 candidates, totalling 100 fits

=== Cost-Sensitive XGB — VALIDATION ===
              precision    recall  f1-score   support

           0      0.000     0.000     0.000     42368
           1      0.094     0.071     0.081     56660
           2      0.000     0.003     0.001      7151
           3      0.004     0.051     0.007       549
           4      0.000     0.000     0.000      1899
           5      0.002     0.001     0.002      3474
           6      0.000     0.000     0.000      4102
           7      0.000     0.000     0.000         0

    accuracy                          0.035    116203
   macro avg      0.013     0.016     0.011    116203
weighted avg      0.046     0.035     0.040    116203

Accuracy         : 0.0351
Balanced Accuracy: 0.0181
F1 Macro         : 0.0113

Per-Class Recall:
  Class 0: 0.0000
  Class 1: 0.0709
  Class 2: 0.0032
  Class 3: 0.0510
  Class 4: 0.0000
  Class 5: 0.0014
  Class 6: 0.0000

Confusion Matrix:
[[  

## 8. Multiple Random Seeds-----------------------------------------------------

In [ ]:
# Run the two strongest models across multiple random seeds
# This tests whether the results are stable or just lucky with seed=42
# A good model should perform consistently regardless of the data split

SEEDS = [42, 0, 7, 123, 999]
seed_f1_smote_rf  = []

# Reuse best hyperparameters found during tuning above
best_smote_rf_params = smote_rf_search.best_params_

for seed in SEEDS:
    # Re-split the data with a different random seed each time
    Xtrv, Xte, ytrv, yte = train_test_split(X, y, test_size=0.20, stratify=y, random_state=seed)
    Xtr, Xv, ytr, yv     = train_test_split(Xtrv, ytrv, test_size=0.25, stratify=ytrv, random_state=seed)

    # SMOTE w RF with best params
    _rf_params = {k.replace('clf__','').replace('smote__k_neighbors','k_neighbors'): v
                  for k, v in best_smote_rf_params.items()}
    k_nb = best_smote_rf_params.get('smote__k_neighbors', 5)
    _rf_clf_params = {k.replace('clf__',''): v for k, v in best_smote_rf_params.items() if k.startswith('clf__')}
    srf = ImbPipeline([
        ('preprocess', preprocess),
        ('smote', SMOTE(k_neighbors=k_nb, random_state=seed)),
        ('clf', RandomForestClassifier(**_rf_clf_params, random_state=seed, n_jobs=-1))
    ])
    srf.fit(Xtr, ytr)
    seed_f1_smote_rf.append(f1_score(yv, srf.predict(Xv), average='macro'))

# Report mean std across seeds
print(f"SMOTE + RF - F1 per seed: {[f'{f:.4f}' for f in seed_f1_smote_rf]}")
print(f"             Avg F1 Macro: {np.mean(seed_f1_smote_rf):.4f} \u00b1 {np.std(seed_f1_smote_rf):.4f}")
print()

SMOTE + RF - F1 per seed: ['0.9256', '0.9246', '0.9276', '0.9244', '0.9243']
             Avg F1 Macro: 0.9253 ± 0.0013

